<a href="https://colab.research.google.com/github/PaulaJ2023/GiraRota/blob/main/Projeto_A3_GiraRota_Algoritmo_de_Dijkstra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto A3 - GiraRota - Algoritmo de Dijkstra

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# =========================================================
# 1. ESTILO VISUAL DOS WIDGETS (Arredondados e Legíveis)
# =========================================================
display(HTML("""
<style>
.widget-label { font-weight: bold; color: #ffffff !important; font-size: 13px !important; text-shadow: 0 1px 2px rgba(0,0,0,0.3); }
.p-Widget.jupyter-widgets { font-family: 'Poppins', sans-serif; }

select, input, .jupyter-widgets select, .jupyter-widgets input {
    border-radius: 14px !important;
    padding: 6px 12px !important;
    border: 1.5px solid #ffd166 !important;
    background-color: #2b2b2b !important;
    color: #ffffff !important;
}
</style>
"""))

# =========================================================
# 2. CONFIGURAÇÃO DO GRAFO E DADOS DO PROJETO
# =========================================================
G = nx.Graph()

arestas = [
    ('Residência', 'Praça Central', 4),
    ('Residência', 'Av. das Flores', 2),
    ('Praça Central', 'Av. das Flores', 1),
    ('Praça Central', 'Rua do Sol', 5),
    ('Av. das Flores', 'Rua do Sol', 3),
    ('Praça Central', 'Universidade', 15),
    ('Rua do Sol', 'Rua do Comércio', 2),
    ('Av. das Flores', 'Rua do Comércio', 6),
    ('Rua do Comércio', 'Universidade', 3)
]

G.add_weighted_edges_from(arestas)

pos = {
    'Residência': (1.2, 4.8),
    'Av. das Flores': (0, 2.8),
    'Praça Central': (4.2, 3.8),
    'Rua do Sol': (4.2, 1.8),
    'Rua do Comércio': (1.8, 0.6),
    'Universidade': (6.2, 0.6)
}

rotulos_formatados = {
    'Residência': 'Residência',
    'Av. das Flores': 'Av. das\nFlores',
    'Praça Central': 'Praça\nCentral',
    'Rua do Sol': 'Rua do\nSol',
    'Rua do Comércio': 'Rua do\nComércio',
    'Universidade': 'Universidade'
}

# =========================================================
# 3. COMPONENTES DO PAINEL INTERATIVO
# =========================================================
lbl_titulo = widgets.HTML(
    "<div style='background: linear-gradient(135deg, #ffd166, #ffb703); padding: 14px 20px; border-radius: 16px; color: #3d2614; font-weight: bold; font-size: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15); margin-bottom: 14px;'>"
    "🌻 <b>GiraRota — Painel de Controle e Segurança do Trajeto</b>"
    "</div>"
)

dd_origem = widgets.Dropdown(options=list(pos.keys()), value='Residência', description='<b>Partida:</b>', style={'description_width': '75px'})
dd_destino = widgets.Dropdown(options=list(pos.keys()), value='Universidade', description='<b>Destino:</b>', style={'description_width': '75px'})

dd_bloq_de = widgets.Dropdown(options=list(pos.keys()), value='Praça Central', description='<b>Bloq. De:</b>', style={'description_width': '75px'})
dd_bloq_para = widgets.Dropdown(options=list(pos.keys()), value='Rua do Sol', description='<b>Bloq. Para:</b>', style={'description_width': '75px'})

btn_adicionar = widgets.Button(
    description='➕ Adicionar Bloqueio',
    layout=widgets.Layout(width='165px', height='38px', border_radius='14px'),
    style={'button_color': '#06d6a0', 'font_weight': 'bold'}
)

btn_limpar = widgets.Button(
    description='🗑️ Limpar Todos',
    layout=widgets.Layout(width='140px', height='38px', border_radius='14px'),
    style={'button_color': '#ff4d6d', 'font_weight': 'bold'}
)

out_status = widgets.Output(layout=widgets.Layout(margin='10px 0 0 0'))
out_lista_bloqueios = widgets.Output(layout=widgets.Layout(margin='8px 0'))

vias_bloqueadas_usuario = set()

# =========================================================
# 4. FUNÇÃO DE ATUALIZAÇÃO DA SIMULAÇÃO (DIJKSTRA + MAPAS)
# =========================================================
def atualizar_simulacao(change=None):
    with out_status, out_lista_bloqueios:
        out_status.clear_output(wait=True)
        out_lista_bloqueios.clear_output(wait=True)

        origem = dd_origem.value
        destino = dd_destino.value

        # Recria grafo limpo
        G_sim = nx.Graph()
        G_sim.add_weighted_edges_from(arestas)

        # Aplica bloqueios (peso infinito)
        for u, v in vias_bloqueadas_usuario:
            if G_sim.has_edge(u, v):
                G_sim[u][v]['weight'] = float('inf')
                G_sim[v][u]['weight'] = float('inf')

        # Processa pares únicos para exibição limpa
        pares_unicos = []
        vistos = set()
        for u, v in sorted(list(vias_bloqueadas_usuario)):
            normal = tuple(sorted([u, v]))
            if normal not in vistos:
                vistos.add(normal)
                pares_unicos.append((u, v))

        # Lista de bloqueios interativa com remoção individual
        if pares_unicos:
            with out_lista_bloqueios:
                display(HTML("<b style='color: #ffd166; font-size: 14px;'>🛡️ Vias Evitadas (Clique para remover individualmente):</b>"))
                for par in pares_unicos:
                    u, v = par
                    btn_del = widgets.Button(
                        description=f"❌ Remover ({u} ➔ {v})",
                        layout=widgets.Layout(width='auto', height='36px', margin='5px 0', border_radius='16px'),
                        style={'button_color': '#2b2b2b', 'font_weight': 'bold', 'text_color': '#ffd166'}
                    )
                    def criar_closure(p):
                        def acao(b):
                            if p in vias_bloqueadas_usuario:
                                vias_bloqueadas_usuario.remove(p)
                            inv = (p[1], p[0])
                            if inv in vias_bloqueadas_usuario:
                                vias_bloqueadas_usuario.remove(inv)
                            atualizar_simulacao()
                        return acao
                    btn_del.on_click(criar_closure(par))
                    display(btn_del)

        # Valida se existe caminho seguro
        arestas_validas = [(u, v, d) for u, v, d in G_sim.edges(data=True) if d['weight'] != float('inf')]
        G_seguro = nx.Graph()
        G_seguro.add_edges_from([(u, v) for u, v, _ in arestas_validas])

        fig, ax = plt.subplots(figsize=(12, 6.5))
        fig.patch.set_facecolor('#fff0f3')
        ax.set_facecolor('#fff0f3')

        tem_caminho = nx.has_path(G_seguro, origem, destino)

        with out_status:
            if tem_caminho:
                melhor_rota = nx.dijkstra_path(G_sim, source=origem, target=destino, weight='weight')
                custo_total = nx.dijkstra_path_length(G_sim, source=origem, target=destino, weight='weight')
                nos_rota = set(melhor_rota)

                display(HTML(f"""
                <div style='background: #ffffff; border-left: 5px solid #06d6a0; padding: 12px 16px; border-radius: 14px; box-shadow: 0 3px 6px rgba(0,0,0,0.08); margin-bottom: 10px; font-family: sans-serif; color: #212529;'>
                    <b>🌻 Status da Corrida Segura:</b><br>
                    📍 <b>Partida:</b> {origem} &nbsp;|&nbsp; 🏁 <b>Destino:</b> {destino}<br>
                    ✅ <b>Melhor Rota:</b> {' ➔ '.join(melhor_rota)}<br>
                    ⭐ <b>Custo Total:</b> {custo_total}
                </div>
                """))

                cor_nos = ['#ffffff' for node in G_sim.nodes()]
                cores_bordas = ['#ff4d6d' if node in nos_rota else '#495057' for node in G_sim.nodes()]
                nx.draw_networkx_nodes(G_sim, pos, ax=ax, node_size=6000, node_color=cor_nos, edgecolors=cores_bordas, linewidths=3)

                arestas_normais = [(u, v) for (u, v, d) in G_sim.edges(data=True) if d['weight'] != float('inf') and not ((u in melhor_rota and v in melhor_rota) and abs(melhor_rota.index(u) - melhor_rota.index(v)) == 1)]
                arestas_otimas = [(melhor_rota[i], melhor_rota[i+1]) for i in range(len(melhor_rota)-1)]
                arestas_bloq = [(u, v) for (u, v, d) in G_sim.edges(data=True) if d['weight'] == float('inf')]

                nx.draw_networkx_edges(G_sim, pos, ax=ax, edgelist=arestas_normais, width=2.5, edge_color='#ced4da', alpha=0.8)
                nx.draw_networkx_edges(G_sim, pos, ax=ax, edgelist=arestas_otimas, width=5.5, edge_color='#06d6a0')
                if arestas_bloq:
                    nx.draw_networkx_edges(G_sim, pos, ax=ax, edgelist=arestas_bloq, width=3.5, edge_color='#e71d36', style='dashed')

                nx.draw_networkx_labels(G_sim, pos, labels=rotulos_formatados, ax=ax, font_size=10, font_weight='bold', font_color='#000000')

                labels = {}
                for u, v, d in G_sim.edges(data=True):
                    if d['weight'] == float('inf'):
                        labels[(u, v)] = ' [BLOQUEADA] \n (Evitada)'
                    else:
                        labels[(u, v)] = f" {d['weight']} "
                nx.draw_networkx_edge_labels(G_sim, pos, edge_labels=labels, font_size=9, font_weight='bold', font_color='#212529', ax=ax)

                ax.set_title(f"GiraRota - Rota Inteligente e Segura\nPartida: {origem} ➔ Destino: {destino} (Custo: {custo_total})", fontsize=14, fontweight='bold', color='#212529', pad=15)

            else:
                display(HTML(f"""
                <div style='background: #fff5f5; border-left: 5px solid #e71d36; padding: 12px 16px; border-radius: 14px; box-shadow: 0 3px 6px rgba(0,0,0,0.08); margin-bottom: 10px; font-family: sans-serif; color: #9b1c1c;'>
                    <b>⚠️ ATENÇÃO - Rota Indisponível:</b><br>
                    As vias bloqueadas isolaram completamente o destino <b>{destino}</b>. Não há caminhos seguros disponíveis com os filtros atuais.
                </div>
                """))

                nx.draw_networkx_nodes(G_sim, pos, ax=ax, node_size=6000, node_color='#ffffff', edgecolors='#495057', linewidths=3)
                nx.draw_networkx_edges(G_sim, pos, ax=ax, edgelist=list(G_sim.edges()), width=2.5, edge_color='#ced4da')
                nx.draw_networkx_labels(G_sim, pos, labels=rotulos_formatados, ax=ax, font_size=10, font_weight='bold', font_color='#000000')
                ax.set_title("GiraRota - Alerta: Destino Inalcançável por Bloqueio de Vias", fontsize=14, fontweight='bold', color='#e71d36', pad=15)

            ax.axis('off')
            plt.tight_layout()
            plt.show()

# =========================================================
# 5. EVENTOS E RENDERIZAÇÃO FINAL DOS WIDGETS
# =========================================================
def on_add_clicked(b):
    par = (dd_bloq_de.value, dd_bloq_para.value)
    par_inverso = (dd_bloq_para.value, dd_bloq_de.value)
    if par[0] != par[1]:
        vias_bloqueadas_usuario.add(par)
        vias_bloqueadas_usuario.add(par_inverso)
    atualizar_simulacao()

def on_clear_clicked(b):
    vias_bloqueadas_usuario.clear()
    atualizar_simulacao()

btn_adicionar.on_click(on_add_clicked)
btn_limpar.on_click(on_clear_clicked)
dd_origem.observe(atualizar_simulacao, names='value')
dd_destino.observe(atualizar_simulacao, names='value')

ui_box = widgets.VBox([
    lbl_titulo,
    widgets.HBox([dd_origem, dd_destino], layout=widgets.Layout(margin='0 0 8px 0')),
    widgets.HBox([dd_bloq_de, dd_bloq_para, btn_adicionar, btn_limpar], layout=widgets.Layout(margin='0 0 8px 0', align_items='center')),
    out_lista_bloqueios,
    out_status
], layout=widgets.Layout(background_color='#36251b', padding='20px', border_radius='22px', border='2px solid #ffd166', box_shadow='0 8px 16px rgba(0,0,0,0.25)'))

display(ui_box)
atualizar_simulacao()

# Pseudocodigo

In [ ]:
ALGORITMO GiraRota em pseudocódigo
INÍCIO
    // 1. Definição de Parâmetros e Locais
    ORIGEM_BLOQUEIO ← "Praça Central"
    DESTINO_BLOQUEIO ← "Rua do Sol"
    PONTO_ORIGEM ← "Residência"
    PONTO_DESTINO ← "Universidade"

    // 2. Inicialização do Grafo
    Criar Grafo Vazio (G)

    // 3. Adição das Arestas e Pesos (Custo Composto)
    AdicionarAresta(G, 'Residência', 'Praça Central', peso = 4)
    AdicionarAresta(G, 'Residência', 'Av. das Flores', peso = 2)
    AdicionarAresta(G, 'Praça Central', 'Av. das Flores', peso = 1)
    AdicionarAresta(G, 'Praça Central', 'Rua do Sol', peso = 5) // Via que pode ser bloqueada
    AdicionarAresta(G, 'Av. das Flores', 'Rua do Sol', peso = 3)
    AdicionarAresta(G, 'Praça Central', 'Universidade', peso = 15)
    AdicionarAresta(G, 'Rua do Sol', 'Rua do Comércio', peso = 2)
    AdicionarAresta(G, 'Av. das Flores', 'Rua do Comércio', peso = 6)
    AdicionarAresta(G, 'Rua do Comércio', 'Universidade', peso = 3)

    // 4. Aplicação da Regra de Segurança (Bloqueio de Via)
    SE ExisteAresta(G, ORIGEM_BLOQUEIO, DESTINO_BLOQUEIO) ENTÃO
        DefinirPesoAresta(G, ORIGEM_BLOQUEIO, DESTINO_BLOQUEIO, INFINITO)
        DefinirPesoAresta(G, DESTINO_BLOQUEIO, ORIGEM_BLOQUEIO, INFINITO) // Via bidirecional
    FIM_SE

    // 5. Cálculo da Rota Segura com o Algoritmo de Dijkstra
    melhor_rota ← ExecutarDijkstraCaminho(G, origem = PONTO_ORIGEM, destino = PONTO_DESTINO)
    custo_total ← ExecutarDijkstraCusto(G, origem = PONTO_ORIGEM, destino = PONTO_DESTINO)

    // 6. Exibição dos Resultados Textuais
    Imprimir("=== GIRAROTA - ROTA SEGURA ===")
    Imprimir("Partida: ", PONTO_ORIGEM, " | Destino: ", PONTO_DESTINO)
    Imprimir("Melhor Rota: ", melhor_rota)
    Imprimir("Custo Total: ", custo_total)

    // 7. Visualização Gráfica (Resumo da Interface)
    ConfigurarJanelaGrafica(cor_fundo = "Rosa Suave")
    DesenharMapaComRotasDestacadas(G, melhor_rota, arestas_bloqueadas)
    MostrarTela()
FIM
